In [1]:
!wget -q https://github.com/ai-dock/llama.cpp-cuda/releases/download/v0.4.0/llama.cpp-v0.4.0-cuda-12.8-amd64.tar.gz
!tar -xzf llama.cpp-v0.4.0-cuda-12.8-amd64.tar.gz
!find . -maxdepth 3 -type f -name "llama-cli" -o -name "llama-server"

./cuda-12.8/llama-cli
./cuda-12.8/llama-server


In [2]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com bookworm main
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 https://ngrok-agent.s3.amazonaws.com bookworm InRelease [20.3 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]m
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://cli.github.com/packages stable/main amd64 Packages [359 B]      
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]m
Get:12 https://developer.

In [3]:
!ngrok config add-authtoken <API-KEY>

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!export LD_LIBRARY_PATH=$PWD/cuda-12.8:$LD_LIBRARY_PATH && \
./cuda-12.8/llama-server \
  -hf unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL \
  -ngl auto \
  -sm tensor \
  --flash-attn on \
  --cache-type-k q8_0 \
  --cache-type-v q8_0 \
  --ctx-size 131072 \
  -b 1024 \
  -ub 512 \
  -np 1 \
  --reasoning-preserve \
  --port 8080

0.00.525.516 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.526.274 W srv  llama_server: -----------------
0.00.526.282 W srv  llama_server: CORS is set to allow all origins ('*') and no API key is set
0.00.526.282 W srv  llama_server: this can be a security risk (cross-origin attacks)
0.00.526.283 W srv  llama_server: more info: https://github.com/ggml-org/llama.cpp/pull/25655
0.00.526.283 W srv  llama_server: -----------------
0.00.528.191 I srv    load_model: loading model 'unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL'
0.00.833.840 W common_fit_params: failed to fit params to free device memory: llama_params_fit is not implemented for SPLIT_MODE_TENSOR, abort
0.10.359.480 I cmn          init: llama threadpool init, n_threads = 2
0.10.770.525 W load_hparams: Qwen-VL models require at minimum 1024 image tokens to function correctly on grounding tasks
0.10.770.535 W load_hparams: if you encounter problems with accuracy, try adding --image

In [6]:
import os
import subprocess
import time

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    os.path.abspath("./cuda-12.8")
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)

cmd = [
    "./cuda-12.8/llama-server",
    "-hf", "unsloth/Qwen3.8-27B-GGUF:UD-Q4_K_M",
    "-ngl", "auto",
    "-sm", "tensor",
    "--flash-attn", "on",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "--ctx-size", "131072",
    "-b", "1024",
    "-ub", "512",
    "-np", "1",
    "--reasoning-preserve",
    "--port", "8080",
]

log_path = "llama.log"

print("🚀 Iniciando llama-server...")
print("📦 Cargando modelo...")
print("🧠 Preparando CUDA...")
print("⚡ Configurando Flash Attention...")
print("💾 Configurando KV cache...")
print("📚 Preparando contexto de 65K...")
print("⏳ Esperando a que el servidor esté listo...\n")

# Abrimos el log en modo escritura
log = open(log_path, "w", buffering=1)

process = subprocess.Popen(
    cmd,
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"🔄 llama-server iniciado (PID: {process.pid})")
print("📖 Leyendo logs...\n")

# Mensajes "animados"
mensajes = [
    "⏳ Cargando...",
    "🔧 Inicializando servidor...",
    "🧠 Cargando pesos del modelo...",
    "⚡ Inicializando GPU...",
    "📦 Preparando memoria...",
    "🚀 Levantando servidor...",
]

mensaje_idx = 0
ultima_posicion = 0

while True:
    # Si el proceso murió antes de estar listo
    if process.poll() is not None:
        log.close()

        print("\n❌ llama-server terminó antes de estar listo.")
        print(f"📄 Revisa el log: {log_path}")

        with open(log_path, "r", errors="replace") as f:
            print("\n--- Últimos logs ---")
            print(f.read()[-3000:])

        raise SystemExit(1)

    # Leer lo nuevo que haya aparecido en el log
    try:
        with open(log_path, "r", errors="replace") as f:
            f.seek(ultima_posicion)
            nuevos_logs = f.read()
            ultima_posicion = f.tell()

        if nuevos_logs:
            # Buscar el mensaje de servidor listo
            if "listening on http://127.0.0.1:8080" in nuevos_logs:
                print("\n" + "=" * 60)
                print("🟢 LISTO")
                print("🌐 llama-server está escuchando en:")
                print("   http://127.0.0.1:8080")
                print(f"🆔 PID: {process.pid}")
                print("📚 Contexto: 65,536 tokens")
                print("=" * 60)

                # Cerramos nuestro handle del log.
                # NO matamos llama-server: queda en segundo plano.
                log.close()

                print("\n✅ El proceso queda corriendo en segundo plano.")
                break

    except FileNotFoundError:
        pass

    # Mostrar progreso cada ~2 segundos
    print(mensajes[mensaje_idx], end="\r", flush=True)
    mensaje_idx = (mensaje_idx + 1) % len(mensajes)

    time.sleep(2)

🚀 Iniciando llama-server...
📦 Cargando modelo...
🧠 Preparando CUDA...
⚡ Configurando Flash Attention...
💾 Configurando KV cache...
📚 Preparando contexto de 65K...
⏳ Esperando a que el servidor esté listo...

🔄 llama-server iniciado (PID: 1039)
📖 Leyendo logs...

📦 Preparando memoria...delo...
🟢 LISTO
🌐 llama-server está escuchando en:
   http://127.0.0.1:8080
🆔 PID: 1039
📚 Contexto: 65,536 tokens

✅ El proceso queda corriendo en segundo plano.


In [13]:
!pkill -9 -f llama-server

In [5]:
import subprocess

subprocess.Popen(
    ["ngrok", "http", "8080"],
    stdout=open("ngrok.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("🚀 ngrok corriendo en segundo plano")

🚀 ngrok corriendo en segundo plano


In [39]:
!pkill -9 -f ngrok

In [7]:
!ngrok http 8080

7=ngrok                                                           (Ctrl+C to quit)                                                                                Session Status                connecting                                        Version                       3.39.11                                           Web Interface                 http://127.0.0.1:4040                                                                                                             Connections                   ttl     opn     rt1     rt5     p50     p90                                     0       0       0.00    0.00    0.00    0.00                                                                                                                                                                                                                                                                                                                                                                        

In [12]:
!cat llama.log

0.00.524.537 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.525.145 W srv  llama_server: -----------------
0.00.525.152 W srv  llama_server: CORS is set to allow all origins ('*') and no API key is set
0.00.525.152 W srv  llama_server: this can be a security risk (cross-origin attacks)
0.00.525.153 W srv  llama_server: more info: https://github.com/ggml-org/llama.cpp/pull/25655
0.00.525.153 W srv  llama_server: -----------------
0.00.526.962 I srv    load_model: loading model 'unsloth/Qwen3.8-27B-GGUF:UD-Q4_K_M'
0.00.819.789 W common_fit_params: failed to fit params to free device memory: llama_params_fit is not implemented for SPLIT_MODE_TENSOR, abort
0.01.647.916 W model has unused tensor blk.64.attn_norm.weight (size = 20480 bytes) -- ignoring
0.01.647.931 W model has unused tensor blk.64.post_attention_norm.weight (size = 20480 bytes) -- ignoring
0.01.647.945 W model has unused tensor blk.64.attn_q.weight (size = 51609600 bytes) 

In [ ]:
!rm llama.log

In [ ]:
!nvidia-smi